# Rich Output Formats Comparison

Comparing Rich's different output surfaces: terminal markup, HTML export, SVG rendering, and live recording.

## Setup

In [ ]:
from io import StringIO
from rich.console import Console
from rich.markdown import Markdown
from rich.table import Table
from rich.panel import Panel
from rich.text import Text

## 1. Terminal Markup

Rich's `[bold red]text[/bold red]` syntax renders styled text directly in the terminal.

In [ ]:
console = Console()

console.print("[bold green]Success![/bold green] All tests passed.")
console.print("[red]Error:[/red] File not found.")
console.print("[bold blue on white] Highlighted [/bold blue on white] background.")
console.print()

# Nested styles
console.print("[bold]Bold [italic]and italic[/italic] text[/bold]")

## 2. HTML Export

`console.export_html()` produces a self-contained HTML page that mirrors terminal styling. Useful for CI artifacts or web-based log viewers.

In [ ]:
html_console = Console(file=StringIO(), force_terminal=True, width=80)

table = Table(title="Dependency Status")
table.add_column("Package", style="cyan")
table.add_column("Version", style="green")
table.add_column("Status", style="bold")
table.add_row("ruff", "0.16.5", "[green]OK[/green]")
table.add_row("mypy", "1.15.0", "[yellow]Update available[/yellow]")
table.add_row("pytest", "8.3.5", "[green]OK[/green]")

html_console.print(table)
html_output = html_console.file.getvalue()

# Show a snippet of the HTML (first 500 chars)
print(html_output[:500])
print(f"\n... ({len(html_output)} total chars)")

## 3. SVG Output

Rich can render panels and text as inline SVG. This is less common than HTML but useful when you need vector graphics in documentation or web pages.

In [ ]:
svg_console = Console(file=StringIO(), force_terminal=True, width=60)

panel = Panel(
    "[bold]Build Summary[/bold]\n\n"
    "Tests: [green]42 passed[/green]\n"
    "Lint: [green]0 errors[/green]\n"
    "Type check: [yellow]3 warnings[/yellow]",
    title="[blue]CI Pipeline[/blue]",
    border_style="blue",
)

svg_console.print(panel)
svg_output = svg_console.file.getvalue()

# SVG output starts with <svg tag when available
print(f"Output length: {len(svg_output)} chars")
print(f"Starts with SVG tag: {svg_output.strip().startswith('<svg')}")

## 4. Live Display Recording

`rich.live` captures the sequence of terminal frames. Useful for replaying build output or debugging CI rendering.

In [ ]:
from rich.live import Live
from rich.table import Table as LiveTable

def build_step_table(step: int, total: int) -> LiveTable:
    """Build a table showing the current pipeline step."""
    tbl = LiveTable(title="Pipeline Progress")
    tbl.add_column("Step", style="cyan")
    tbl.add_column("Status", style="bold")
    for i in range(1, total + 1):
        if i < step:
            tbl.add_row(f"Step {i}", "[green]Done[/green]")
        elif i == step:
            tbl.add_row(f"Step {i}", "[yellow]Running...[/yellow]")
        else:
            tbl.add_row(f"Step {i}", "[dim]Pending[/dim]")
    return tbl

# Capture frames into a list
frames = []
with Live(build_step_table(1, 4), console=Console(file=StringIO(), force_terminal=True)) as live:
    for step in range(1, 5):
        live.update(build_step_table(step, 4))
        # In real use, work happens here

print(f"Live display captured {len(frames)} frames")
print("Live display ran successfully — frames are in-memory.")

## Comparison Table

| Format | Use case | Interactive | Portable |
|--------|----------|-------------|----------|
| Terminal markup | Developer CLI output | Yes (terminal only) | No |
| HTML export | CI artifacts, log viewers | No (static) | Yes |
| SVG output | Documentation, web embedding | No (static) | Yes |
| Live recording | Debug, replay | Yes (terminal only) | Partial (JSON frames) |

## Verify

Run each cell to confirm:
1. Terminal markup prints styled text.
2. HTML export produces a string starting with `<!DOCTYPE html>` or `<pre>`.
3. SVG output produces a string with `<svg` tag.
4. Live display completes without errors.